In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 220)

DATA_DIR = Path("data")

OUT_M = Path("m_data.csv")
OUT_W = Path("w_data.csv")

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Nie znaleziono folderu: {DATA_DIR.resolve()}")

In [16]:
def read_csv_required(filename: str) -> pd.DataFrame:
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Brakuje pliku: {path}")
    print(f"[read] {filename}")
    return pd.read_csv(path)


def read_csv_optional(filename: str):
    path = DATA_DIR / filename
    if not path.exists():
        print(f"[skip] brak opcjonalnego pliku: {filename}")
        return None
    print(f"[read] {filename}")
    return pd.read_csv(path)

In [17]:
# MEN
m_regular = read_csv_required("MRegularSeasonDetailedResults.csv")
m_tourney = read_csv_required("MNCAATourneyDetailedResults.csv")
m_seeds = read_csv_optional("MNCAATourneySeeds.csv")
m_coaches = read_csv_optional("MTeamCoaches.csv")

m_massey_p1 = read_csv_optional("MMasseyOrdinals_part_1.csv")
m_massey_p2 = read_csv_optional("MMasseyOrdinals_part_2.csv")

# WOMEN
w_regular = read_csv_required("WRegularSeasonDetailedResults.csv")
w_tourney = read_csv_required("WNCAATourneyDetailedResults.csv")
w_seeds = read_csv_optional("WNCAATourneySeeds.csv")

[read] MRegularSeasonDetailedResults.csv
[read] MNCAATourneyDetailedResults.csv
[read] MNCAATourneySeeds.csv
[read] MTeamCoaches.csv
[read] MMasseyOrdinals_part_1.csv
[read] MMasseyOrdinals_part_2.csv
[read] WRegularSeasonDetailedResults.csv
[read] WNCAATourneyDetailedResults.csv
[read] WNCAATourneySeeds.csv


In [18]:
# filtr sezonów
m_regular = m_regular[m_regular["Season"] >= 2003].copy()
m_tourney = m_tourney[m_tourney["Season"] >= 2003].copy()

if m_seeds is not None:
    m_seeds = m_seeds[m_seeds["Season"] >= 2003].copy()

if m_coaches is not None:
    m_coaches = m_coaches[m_coaches["Season"] >= 2003].copy()

if m_massey_p1 is not None:
    m_massey_p1 = m_massey_p1[m_massey_p1["Season"] >= 2003].copy()

if m_massey_p2 is not None:
    m_massey_p2 = m_massey_p2[m_massey_p2["Season"] >= 2003].copy()

w_regular = w_regular[w_regular["Season"] >= 2010].copy()
w_tourney = w_tourney[w_tourney["Season"] >= 2010].copy()

if w_seeds is not None:
    w_seeds = w_seeds[w_seeds["Season"] >= 2010].copy()

In [19]:
# sklejenie Massey part_1 + part_2
m_massey_parts = [df for df in [m_massey_p1, m_massey_p2] if df is not None]

if len(m_massey_parts) > 0:
    m_massey = pd.concat(m_massey_parts, ignore_index=True)
    print("MMassey shape:", m_massey.shape)
else:
    m_massey = None
    print("Brak danych MMassey")

MMassey shape: (5819228, 5)


In [20]:
COMMON_GAME_COLS = [
    "Season", "DayNum", "WTeamID", "WScore", "LTeamID", "LScore", "WLoc", "NumOT"
]

DETAILED_EXTRA_COLS = [
    "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF",
    "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF"
]

ALL_GAME_COLS = COMMON_GAME_COLS + DETAILED_EXTRA_COLS


def add_game_id(df: pd.DataFrame, gender: str) -> pd.DataFrame:
    out = df.copy()
    low_id = np.minimum(out["WTeamID"].astype(int), out["LTeamID"].astype(int))
    high_id = np.maximum(out["WTeamID"].astype(int), out["LTeamID"].astype(int))
    out["GameID"] = (
        gender + "_"
        + out["Season"].astype(str) + "_"
        + out["DayNum"].astype(str) + "_"
        + low_id.astype(str) + "_"
        + high_id.astype(str)
    )
    return out


def add_basic_flags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["NeutralSite"] = (out["WLoc"] == "N").astype("int8")
    out["WHome"] = (out["WLoc"] == "H").astype("int8")
    out["WAway"] = (out["WLoc"] == "A").astype("int8")
    out["LHome"] = (out["WLoc"] == "A").astype("int8")
    out["LAway"] = (out["WLoc"] == "H").astype("int8")
    out["ScoreDiff"] = pd.to_numeric(out["WScore"], errors="coerce") - pd.to_numeric(out["LScore"], errors="coerce")
    out["TotalScore"] = pd.to_numeric(out["WScore"], errors="coerce") + pd.to_numeric(out["LScore"], errors="coerce")
    return out


def add_seed_info(df: pd.DataFrame, seeds: pd.DataFrame, side_prefix: str) -> pd.DataFrame:
    if seeds is None:
        return df

    meta = seeds.rename(columns={"TeamID": f"{side_prefix}TeamID", "Seed": f"{side_prefix}Seed"}).copy()
    meta = meta[[c for c in ["Season", f"{side_prefix}TeamID", f"{side_prefix}Seed"] if c in meta.columns]]

    out = df.merge(meta, on=["Season", f"{side_prefix}TeamID"], how="left")

    seed_col = f"{side_prefix}Seed"
    if seed_col in out.columns:
        seed_str = out[seed_col].astype("string")
        out[f"{side_prefix}SeedRegion"] = seed_str.str.extract(r"^([A-Z])", expand=False)
        out[f"{side_prefix}SeedNum"] = pd.to_numeric(
            seed_str.str.extract(r"^[A-Z](\d{2})", expand=False),
            errors="coerce"
        ).astype("Int64")

    return out

In [21]:
def build_massey_snapshot(massey: pd.DataFrame):
    if massey is None:
        return None

    required = {"Season", "RankingDayNum", "SystemName", "TeamID", "OrdinalRank"}
    if not required.issubset(set(massey.columns)):
        return None

    mm = massey.copy()
    mm = mm[list(required)].dropna(subset=["Season", "RankingDayNum", "TeamID", "OrdinalRank"]).copy()

    mm["Season"] = mm["Season"].astype(int)
    mm["RankingDayNum"] = mm["RankingDayNum"].astype(int)
    mm["TeamID"] = mm["TeamID"].astype(int)
    mm["OrdinalRank"] = pd.to_numeric(mm["OrdinalRank"], errors="coerce")
    mm = mm.dropna(subset=["OrdinalRank"])

    agg = (
        mm.groupby(["Season", "RankingDayNum", "TeamID"], as_index=False)
          .agg(
              MasseyMeanRank=("OrdinalRank", "mean"),
              MasseyMedianRank=("OrdinalRank", "median"),
              MasseyBestRank=("OrdinalRank", "min"),
              MasseyWorstRank=("OrdinalRank", "max"),
              MasseySystemCount=("OrdinalRank", "size"),
              MasseyTop10Count=("OrdinalRank", lambda s: int((s <= 10).sum())),
              MasseyTop25Count=("OrdinalRank", lambda s: int((s <= 25).sum())),
          )
    )

    preferred_systems = ["POM", "SAG", "MOR", "RPI", "WLK", "DOK", "COL", "MAS", "BOB"]
    available = [s for s in preferred_systems if s in set(mm["SystemName"].unique())]

    if available:
        pivot = (
            mm[mm["SystemName"].isin(available)]
            .pivot_table(
                index=["Season", "RankingDayNum", "TeamID"],
                columns="SystemName",
                values="OrdinalRank",
                aggfunc="first"
            )
            .reset_index()
        )

        pivot.columns = [
            c if c in ["Season", "RankingDayNum", "TeamID"] else f"Massey_{c}"
            for c in pivot.columns
        ]

        agg = agg.merge(pivot, on=["Season", "RankingDayNum", "TeamID"], how="left")

    return agg


m_massey_snapshot = build_massey_snapshot(m_massey)

if m_massey_snapshot is not None:
    print("Massey snapshot shape:", m_massey_snapshot.shape)
    display(m_massey_snapshot.head())
else:
    print("Brak snapshotu Massey")

Massey snapshot shape: (269651, 19)


,Season,RankingDayNum,TeamID,MasseyMeanRank,MasseyMedianRank,MasseyBestRank,MasseyWorstRank,MasseySystemCount,MasseyTop10Count,MasseyTop25Count,Massey_BOB,Massey_COL,Massey_DOK,Massey_MAS,Massey_MOR,Massey_POM,Massey_RPI,Massey_SAG,Massey_WLK
0,2003,35,1102,159.0,159.0,159,159,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2003,35,1103,229.0,229.0,229,229,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2003,35,1104,12.0,12.0,12,12,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2003,35,1105,314.0,314.0,314,314,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2003,35,1106,260.0,260.0,260,260,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
def add_latest_massey(df: pd.DataFrame, massey_snapshot: pd.DataFrame, side_prefix: str) -> pd.DataFrame:
    if massey_snapshot is None:
        return df

    left = df.copy()
    right = massey_snapshot.copy()

    left[f"{side_prefix}RankKey"] = (
        left["Season"].astype(str) + "_" + left[f"{side_prefix}TeamID"].astype(str)
    )
    right["RankKey"] = (
        right["Season"].astype(str) + "_" + right["TeamID"].astype(str)
    )

    rename_map = {}
    for col in right.columns:
        if col in ["Season", "TeamID", "RankKey"]:
            continue
        if col == "RankingDayNum":
            rename_map[col] = f"{side_prefix}MasseyDayNum"
        else:
            rename_map[col] = f"{side_prefix}{col}"

    right = right.rename(columns=rename_map)

    right_sort_col = f"{side_prefix}MasseyDayNum"

    keep_cols = ["RankKey", right_sort_col]
    keep_cols += [
        c for c in right.columns
        if c.startswith(side_prefix) and c != right_sort_col
    ]

    right = right[keep_cols].copy()

    left = left.sort_values(["DayNum", f"{side_prefix}RankKey", "GameID"]).reset_index(drop=True)
    right = right.sort_values([right_sort_col, "RankKey"]).reset_index(drop=True)

    out = pd.merge_asof(
        left,
        right,
        left_on="DayNum",
        right_on=right_sort_col,
        left_by=f"{side_prefix}RankKey",
        right_by="RankKey",
        direction="backward"
    )

    out = out.drop(columns=[f"{side_prefix}RankKey", "RankKey"], errors="ignore")
    return out.sort_values(["Season", "DayNum", "GameID"]).reset_index(drop=True)


def add_active_coach(df: pd.DataFrame, coaches: pd.DataFrame, side_prefix: str) -> pd.DataFrame:
    if coaches is None:
        return df

    required = {"Season", "TeamID", "FirstDayNum", "LastDayNum", "CoachName"}
    if not required.issubset(set(coaches.columns)):
        return df

    left = df.copy()
    right = coaches.copy()

    left[f"{side_prefix}CoachKey"] = (
        left["Season"].astype(str) + "_" + left[f"{side_prefix}TeamID"].astype(str)
    )
    right[f"{side_prefix}CoachKey"] = (
        right["Season"].astype(str) + "_" + right["TeamID"].astype(str)
    )

    right = right.rename(columns={
        "FirstDayNum": f"{side_prefix}CoachStartDayNum",
        "LastDayNum": f"{side_prefix}CoachEndDayNum",
        "CoachName": f"{side_prefix}CoachName"
    })

    right = right[[
        f"{side_prefix}CoachKey",
        f"{side_prefix}CoachStartDayNum",
        f"{side_prefix}CoachEndDayNum",
        f"{side_prefix}CoachName"
    ]].copy()

    left = left.sort_values(["DayNum", f"{side_prefix}CoachKey", "GameID"]).reset_index(drop=True)
    right = right.sort_values([f"{side_prefix}CoachStartDayNum", f"{side_prefix}CoachKey"]).reset_index(drop=True)

    out = pd.merge_asof(
        left,
        right,
        left_on="DayNum",
        right_on=f"{side_prefix}CoachStartDayNum",
        left_by=f"{side_prefix}CoachKey",
        right_by=f"{side_prefix}CoachKey",
        direction="backward"
    )

    end_col = f"{side_prefix}CoachEndDayNum"
    start_col = f"{side_prefix}CoachStartDayNum"
    name_col = f"{side_prefix}CoachName"

    if end_col in out.columns:
        invalid = out[end_col].notna() & (out["DayNum"] > out[end_col])
        out.loc[invalid, [start_col, end_col, name_col]] = pd.NA

    out = out.drop(columns=[f"{side_prefix}CoachKey"], errors="ignore")
    return out.sort_values(["Season", "DayNum", "GameID"]).reset_index(drop=True)

In [23]:
def build_gender_dataset(
    gender: str,
    regular: pd.DataFrame,
    tourney: pd.DataFrame,
    seeds: pd.DataFrame = None,
    coaches: pd.DataFrame = None,
    massey_snapshot: pd.DataFrame = None,
) -> pd.DataFrame:

    regular = regular.copy()
    tourney = tourney.copy()

    regular["Gender"] = gender
    regular["GameType"] = "RegularSeason"
    regular["isRegularSeason"] = 1
    regular["isTourney"] = 0
    regular["HasBoxscore"] = 1

    tourney["Gender"] = gender
    tourney["GameType"] = "NCAATourney"
    tourney["isRegularSeason"] = 0
    tourney["isTourney"] = 1
    tourney["HasBoxscore"] = 1

    full = pd.concat([regular, tourney], ignore_index=True)

    full = add_game_id(full, gender)
    full = add_basic_flags(full)

    full = add_seed_info(full, seeds, "W")
    full = add_seed_info(full, seeds, "L")

    if gender == "M":
        full = add_active_coach(full, coaches, "W")
        full = add_active_coach(full, coaches, "L")

        full = add_latest_massey(full, massey_snapshot, "W")
        full = add_latest_massey(full, massey_snapshot, "L")

    front_cols = [
        "GameID", "Gender", "Season", "DayNum", "GameType",
        "isRegularSeason", "isTourney", "HasBoxscore",
        "WTeamID", "LTeamID", "WScore", "LScore", "ScoreDiff", "TotalScore",
        "WLoc", "NeutralSite", "WHome", "WAway", "LHome", "LAway", "NumOT",
        "WSeed", "WSeedNum", "WSeedRegion",
        "LSeed", "LSeedNum", "LSeedRegion",
        "WCoachName", "LCoachName",
        "WMasseyMeanRank", "WMasseyMedianRank", "WMasseyBestRank", "WMasseyWorstRank",
        "LMasseyMeanRank", "LMasseyMedianRank", "LMasseyBestRank", "LMasseyWorstRank",
    ]

    front_cols = [c for c in front_cols if c in full.columns]
    rest_cols = [c for c in full.columns if c not in front_cols]

    full = full[front_cols + rest_cols]
    full = full.sort_values(["Season", "DayNum", "GameID"]).reset_index(drop=True)

    return full

In [24]:
m_data = build_gender_dataset(
    gender="M",
    regular=m_regular,
    tourney=m_tourney,
    seeds=m_seeds,
    coaches=m_coaches,
    massey_snapshot=m_massey_snapshot,
)

w_data = build_gender_dataset(
    gender="W",
    regular=w_regular,
    tourney=w_tourney,
    seeds=w_seeds,
    coaches=None,
    massey_snapshot=None,
)

print("m_data shape:", m_data.shape)
print("w_data shape:", w_data.shape)

display(m_data.head(3))
display(w_data.head(3))

m_data shape: (125480, 93)
w_data shape: (87734, 53)


,GameID,Gender,Season,DayNum,GameType,isRegularSeason,isTourney,HasBoxscore,WTeamID,LTeamID,WScore,LScore,ScoreDiff,TotalScore,WLoc,NeutralSite,WHome,WAway,LHome,LAway,NumOT,WSeed,WSeedNum,WSeedRegion,LSeed,LSeedNum,LSeedRegion,WCoachName,LCoachName,WMasseyMeanRank,WMasseyMedianRank,WMasseyBestRank,WMasseyWorstRank,LMasseyMeanRank,LMasseyMedianRank,LMasseyBestRank,LMasseyWorstRank,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,WCoachStartDayNum,WCoachEndDayNum,LCoachStartDayNum,LCoachEndDayNum,WMasseyDayNum,WMasseySystemCount,WMasseyTop10Count,WMasseyTop25Count,WMassey_BOB,WMassey_COL,WMassey_DOK,WMassey_MAS,WMassey_MOR,WMassey_POM,WMassey_RPI,WMassey_SAG,WMassey_WLK,LMasseyDayNum,LMasseySystemCount,LMasseyTop10Count,LMasseyTop25Count,LMassey_BOB,LMassey_COL,LMassey_DOK,LMassey_MAS,LMassey_MOR,LMassey_POM,LMassey_RPI,LMassey_SAG,LMassey_WLK
0,M_2003_10_1104_1328,M,2003,10,RegularSeason,1,0,1,1104,1328,68,62,6,130,N,1,0,0,0,0,0,Y10,10,Y,W01,1,W,mark_gottfried,kelvin_sampson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27,58,3,14,11,18,14,24,13,23,7,1,22,22,53,2,10,16,22,10,22,8,18,9,2,20,0.0,154.0,0.0,154.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M_2003_10_1272_1393,M,2003,10,RegularSeason,1,0,1,1272,1393,70,63,7,133,N,1,0,0,0,0,0,Z07,7,Z,W03,3,W,john_calipari,jim_boeheim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26,62,8,20,10,19,15,28,16,13,4,4,18,24,67,6,24,9,20,20,25,7,12,8,6,16,0.0,154.0,0.0,154.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M_2003_11_1186_1458,M,2003,11,RegularSeason,1,0,1,1458,1186,81,55,26,136,H,0,1,0,0,1,0,Y05,5,Y,NaN,<NA>,<NA>,bo_ryan,ray_giacoletti,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26,57,6,12,23,27,12,24,12,9,9,3,18,20,46,3,11,12,17,6,22,8,19,4,3,25,0.0,154.0,0.0,154.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,GameID,Gender,Season,DayNum,GameType,isRegularSeason,isTourney,HasBoxscore,WTeamID,LTeamID,WScore,LScore,ScoreDiff,TotalScore,WLoc,NeutralSite,WHome,WAway,LHome,LAway,NumOT,WSeed,WSeedNum,WSeedRegion,LSeed,LSeedNum,LSeedRegion,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,W_2010_11_3102_3394,W,2010,11,RegularSeason,1,0,1,3394,3102,65,46,19,111,H,0,1,0,0,1,0,NaN,<NA>,<NA>,NaN,<NA>,<NA>,25,64,2,18,13,18,21,24,13,18,10,1,16,15,47,5,17,11,20,12,21,8,25,10,0,19
1,W_2010_11_3103_3237,W,2010,11,RegularSeason,1,0,1,3103,3237,63,49,14,112,H,0,1,0,0,1,0,NaN,<NA>,<NA>,NaN,<NA>,<NA>,23,54,5,9,12,19,10,26,14,18,7,0,15,20,54,3,13,6,10,11,27,11,23,7,6,19
2,W_2010_11_3104_3399,W,2010,11,RegularSeason,1,0,1,3104,3399,73,68,5,141,N,1,0,0,0,0,0,NaN,<NA>,<NA>,NaN,<NA>,<NA>,26,62,5,12,16,28,16,31,15,20,5,2,25,25,63,4,21,14,27,14,26,7,20,4,2,27


In [25]:
m_data.to_csv(OUT_M, index=False)
w_data.to_csv(OUT_W, index=False)

print("Zapisano:")
print("-", OUT_M.resolve())
print("-", OUT_W.resolve())

Zapisano:
- C:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026\m_data.csv
- C:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026\w_data.csv


In [26]:
summary = pd.DataFrame({
    "dataset": ["m_data", "w_data"],
    "rows": [len(m_data), len(w_data)],
    "cols": [m_data.shape[1], w_data.shape[1]],
    "first_season": [m_data["Season"].min(), w_data["Season"].min()],
    "last_season": [m_data["Season"].max(), w_data["Season"].max()],
})

display(summary)

print("\nKolumny m_data:")
print(m_data.columns.tolist())

print("\nKolumny w_data:")
print(w_data.columns.tolist())

,dataset,rows,cols,first_season,last_season
0,m_data,125480,93,2003,2026
1,w_data,87734,53,2010,2026



Kolumny m_data:
['GameID', 'Gender', 'Season', 'DayNum', 'GameType', 'isRegularSeason', 'isTourney', 'HasBoxscore', 'WTeamID', 'LTeamID', 'WScore', 'LScore', 'ScoreDiff', 'TotalScore', 'WLoc', 'NeutralSite', 'WHome', 'WAway', 'LHome', 'LAway', 'NumOT', 'WSeed', 'WSeedNum', 'WSeedRegion', 'LSeed', 'LSeedNum', 'LSeedRegion', 'WCoachName', 'LCoachName', 'WMasseyMeanRank', 'WMasseyMedianRank', 'WMasseyBestRank', 'WMasseyWorstRank', 'LMasseyMeanRank', 'LMasseyMedianRank', 'LMasseyBestRank', 'LMasseyWorstRank', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF', 'WCoachStartDayNum', 'WCoachEndDayNum', 'LCoachStartDayNum', 'LCoachEndDayNum', 'WMasseyDayNum', 'WMasseySystemCount', 'WMasseyTop10Count', 'WMasseyTop25Count', 'WMassey_BOB', 'WMassey_COL', 'WMassey_DOK', 'WMassey_MAS', 'WMassey_MOR', 'WMassey_POM', 'WMassey_RPI', 'WMassey_SAG', 'WM